In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import time
# Dataset loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root='./data', train=True, download=True, transform=transform),
    batch_size=64, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root='./data', train=False, transform=transform),
    batch_size=64, shuffle=False
)
class Net_NoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)
class Net_BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        return self.fc3(x)
def train_model(model, optimizer, epochs=5):
    criterion = nn.CrossEntropyLoss()
    start_time = time.time()

    for epoch in range(epochs):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    end_time = time.time()
    return end_time - start_time
def evaluate(model):
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    return 100 * correct / total
model_no_bn = Net_NoBN()
optimizer_no_bn = optim.Adam(model_no_bn.parameters(), lr=0.001)

time_no_bn = train_model(model_no_bn, optimizer_no_bn)
acc_no_bn = evaluate(model_no_bn)
model_bn = Net_BN()
optimizer_bn = optim.Adam(model_bn.parameters(), lr=0.001)

time_bn = train_model(model_bn, optimizer_bn)
acc_bn = evaluate(model_bn)

print("WITHOUT Batch Normalization")
print("Training Time:", round(time_no_bn, 2), "seconds")
print("Test Accuracy:", round(acc_no_bn, 2), "%")

print("\nWITH Batch Normalization")
print("Training Time:", round(time_bn, 2), "seconds")
print("Test Accuracy:", round(acc_bn, 2), "%")

100.0%
100.0%
100.0%
100.0%


WITHOUT Batch Normalization
Training Time: 100.76 seconds
Test Accuracy: 97.24 %

WITH Batch Normalization
Training Time: 112.61 seconds
Test Accuracy: 97.09 %
